# Multimodal UI Testing Agent
**TCS & AMD AI Hackathon 2026 — MULTIMODAL_005**

Detección visual de bugs en interfaces de usuario usando Vision Language Models en AMD MI300X.

**Stack:** PyTorch + ROCm 7.0 · Qwen2-VL-7B · HuggingFace Transformers · Pillow

**Team:** team-1068 · pulido.jose

## Cell 1 — Verificar entorno AMD

In [ ]:
import subprocess, torch

result = subprocess.run(['amd-smi'], capture_output=True, text=True)
print(result.stdout)

print(f'PyTorch:          {torch.__version__}')
print(f'ROCm disponible:  {torch.cuda.is_available()}')
print(f'VRAM total:       {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

import transformers
print(f'Transformers:     {transformers.__version__}')

# Validar versión mínima de transformers para Qwen2-VL
from packaging import version
if version.parse(transformers.__version__) < version.parse('4.45.0'):
    print('WARNING: transformers version may be too old — run Cell 2 first')
else:
    print('Transformers version OK')

## Cell 2 — Instalar/actualizar dependencias

In [ ]:
import sys
!{sys.executable} -m pip install -q --upgrade transformers accelerate torchvision qwen-vl-utils pillow packaging
print('Dependencias instaladas. Verificando imports...')
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
print('Todos los imports OK.')

## Cell 3 — Cargar modelo Qwen2-VL-7B en MI300X

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch, time

MODEL_ID = 'Qwen/Qwen2-VL-7B-Instruct'

print(f'Cargando {MODEL_ID} en AMD MI300X...')
t0 = time.time()

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto'
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f'Modelo cargado en {time.time()-t0:.1f}s')
print(f'VRAM usada: {torch.cuda.memory_allocated(0)/1024**3:.1f} GB')

## Cell 4 — Funciones del agente

In [ ]:
from PIL import Image, ImageDraw
import torch, time

SYSTEM_PROMPT = """You are an expert UI/UX testing agent specialized in visual inspection of web interfaces.

Your reasoning process (always follow this order):
1. OBSERVE: Describe what you literally see in the screenshot
2. IDENTIFY: Name each relevant UI element and its current visual state
3. COMPARE: Check each element against the specification provided
4. CONCLUDE: State your verdict with specific evidence

Output format (always use exactly this structure):
OBSERVATIONS: [what you see]
SPEC VIOLATIONS: [list each violation, or 'None detected']
VERDICT: PASS or FAIL
SEVERITY: NONE / LOW / MEDIUM / HIGH / CRITICAL
SUMMARY: [one sentence]"""


def analyze_ui(image_path: str, spec: str) -> dict:
    t0 = time.time()
    image = Image.open(image_path).convert('RGB')
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': spec}
            ]
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False
        )
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    response = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]
    latency = time.time() - t0
    result = parse_verdict(response)
    result['latency_s'] = round(latency, 2)
    result['raw'] = response
    result['tokens_input'] = inputs.input_ids.shape[1]
    return result


def parse_verdict(text: str) -> dict:
    result = {'verdict': 'UNKNOWN', 'severity': 'UNKNOWN', 'summary': '', 'observations': '', 'violations': ''}
    for line in text.split('\n'):
        line = line.strip()
        if line.startswith('VERDICT:'):
            v = line.replace('VERDICT:', '').strip().upper()
            result['verdict'] = 'PASS' if 'PASS' in v else 'FAIL'
        elif line.startswith('SEVERITY:'):
            result['severity'] = line.replace('SEVERITY:', '').strip()
        elif line.startswith('SUMMARY:'):
            result['summary'] = line.replace('SUMMARY:', '').strip()
        elif line.startswith('OBSERVATIONS:'):
            result['observations'] = line.replace('OBSERVATIONS:', '').strip()
        elif line.startswith('SPEC VIOLATIONS:'):
            result['violations'] = line.replace('SPEC VIOLATIONS:', '').strip()
    return result


def annotate_image(image_path: str, result: dict, output_path: str):
    img = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    verdict = result['verdict']
    color = (46, 184, 122) if verdict == 'PASS' else (224, 60, 60)
    draw.rectangle([0, 0, img.width, 60], fill=color)
    label = f"  {verdict} — {result['severity']}  |  {result['summary'][:80]}"
    draw.text((10, 20), label, fill='white')
    draw.rectangle([0, 0, img.width-1, img.height-1], outline=color, width=6)
    img.save(output_path)
    print(f'Imagen anotada guardada: {output_path}')


print('Funciones del agente cargadas.')

## Cell 5 — Tomar screenshots con Playwright

In [ ]:
import sys
!{sys.executable} -m pip install -q playwright
!playwright install chromium --with-deps

In [ ]:
import subprocess, os, sys

os.makedirs('/workspace/shared/screenshots', exist_ok=True)
os.makedirs('/workspace/shared/annotated', exist_ok=True)

script = '''
import asyncio
from playwright.async_api import async_playwright

async def take_screenshots():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page(viewport={"width": 1280, "height": 800})
        for case_id in ["case1", "case2", "case3"]:
            await page.goto("file:///workspace/shared/demo-app.html")
            await page.evaluate(f"""
                document.querySelectorAll(".page").forEach(p => p.classList.remove("active"));
                document.querySelectorAll(".nav-tab").forEach(t => t.classList.remove("active"));
                document.getElementById("{case_id}").classList.add("active");
            """)
            await page.wait_for_timeout(500)
            await page.screenshot(path=f"/workspace/shared/screenshots/{case_id}.png", full_page=False)
            print(f"Screenshot guardado: {case_id}.png")
        await browser.close()

asyncio.run(take_screenshots())
'''

with open('/workspace/shared/take_screenshots.py', 'w') as f:
    f.write(script)

result = subprocess.run([sys.executable, '/workspace/shared/take_screenshots.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

## Cell 6 — Specs de cada caso

In [ ]:
SPECS = {
    'case1': """FORM: NexaBank login form
SPEC: The 'Sign In' button MUST be ENABLED (dark background, white text) when:
  1. Email field contains a valid email format (x@x.x)
  2. Password field contains 8+ characters
Both fields currently show green borders indicating valid input.
Question: Is the Sign In button state correct?""",

    'case2': """FORM: NexaBank transfer form
SPEC: Amount field MUST only accept values greater than $0.01.
When a NEGATIVE value is entered:
  1. Field MUST show red border
  2. Error message MUST appear: 'Amount must be greater than zero'
  3. 'Confirm Transfer' button MUST be DISABLED
Current amount field value is -500.
SEVERITY NOTE: This is a financial application — negative amounts are CRITICAL.
Question: Is the form correctly validating the negative amount?""",

    'case3': """FORM: NexaBank login form — correctly implemented version
SPEC: Sign In button MUST be ENABLED when both fields are valid.
Email field shows green border with valid email.
Password field has value entered.
Question: Is this login form correctly implemented? Verify all elements."""
}
print('Specs cargadas para los 3 casos.')

## Cell 7 — Ejecutar el agente en los 3 casos

In [ ]:
results = {}

for case_id, spec in SPECS.items():
    print(f'\n{"="*60}')
    print(f'Analizando {case_id}...')
    image_path = f'/workspace/shared/screenshots/{case_id}.png'
    result = analyze_ui(image_path, spec)
    results[case_id] = result
    annotate_image(image_path, result, f'/workspace/shared/annotated/{case_id}_annotated.png')
    print(f'VERDICT:  {result["verdict"]}')
    print(f'SEVERITY: {result["severity"]}')
    print(f'SUMMARY:  {result["summary"]}')
    print(f'LATENCY:  {result["latency_s"]}s')
    print(f'TOKENS:   {result["tokens_input"]} input tokens')
    print(f'\nRAW OUTPUT:\n{result["raw"]}')

## Cell 8 — Métricas para Slide 4

In [ ]:
import json

expected = {'case1': 'FAIL', 'case2': 'FAIL', 'case3': 'PASS'}

print('='*60)
print('MÉTRICAS — SLIDE 4')
print('='*60)
print(f'Modelo: Qwen2-VL-7B-Instruct')
print(f'Hardware: AMD Instinct MI300X (192GB VRAM)')
print(f'ROCm: 7.0.0 | PyTorch: {torch.__version__}')
print(f'VRAM usada: {torch.cuda.memory_allocated(0)/1024**3:.1f} GB')
print()

correct = 0
total_latency = 0
total_tokens = 0

for case_id, result in results.items():
    is_correct = result['verdict'] == expected[case_id]
    correct += int(is_correct)
    total_latency += result['latency_s']
    total_tokens += result['tokens_input']
    status = '✓' if is_correct else '✗'
    print(f'{status} {case_id}: {result["verdict"]} (expected {expected[case_id]}) — {result["latency_s"]}s')

print()
print(f'Accuracy:          {correct}/{len(results)} ({correct/len(results)*100:.0f}%)')
print(f'Avg latency:       {total_latency/len(results):.1f}s per screenshot')
print(f'Avg tokens input:  {total_tokens//len(results)} tokens per scenario')
print(f'End-to-end (3 cases): {total_latency:.1f}s')

metrics = {
    'model': 'Qwen2-VL-7B-Instruct',
    'hardware': 'AMD Instinct MI300X',
    'rocm_version': '7.0.0',
    'accuracy': f'{correct}/{len(results)}',
    'avg_latency_s': round(total_latency/len(results), 2),
    'avg_tokens_input': total_tokens//len(results),
    'vram_used_gb': round(torch.cuda.memory_allocated(0)/1024**3, 1),
    'results': {
        k: {'verdict': v['verdict'], 'severity': v['severity'],
            'latency_s': v['latency_s'], 'summary': v['summary']}
        for k, v in results.items()
    }
}

with open('/workspace/shared/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('\nMétricas guardadas en /workspace/shared/metrics.json')

## Cell 9 — Mostrar imágenes anotadas

In [ ]:
from IPython.display import display, Image as IPImage
import os

for case_id in ['case1', 'case2', 'case3']:
    path = f'/workspace/shared/annotated/{case_id}_annotated.png'
    if os.path.exists(path):
        print(f'\n--- {case_id} ---')
        display(IPImage(path, width=900))

## Cell 10 — Listar archivos generados

In [ ]:
import os

print('Archivos en /workspace/shared:')
for root, dirs, files in os.walk('/workspace/shared'):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024
        print(f'  {path} ({size:.1f} KB)')

print('\nRecordá descargar todo antes del 15 de junio a medianoche.')